# EmbedMed

## Setup

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")

PDF_PATH = "assets/hypertension-in-adults-diagnosis-and-management.pdf"

print("API Key loaded:", bool(GROQ_API_KEY))
print("Using PDF:", PDF_PATH)

API Key loaded: True
Using PDF: assets/hypertension-in-adults-diagnosis-and-management.pdf


In [6]:
from langchain_community.document_loaders import PyPDFLoader

DOC_ID = "NICE-NG136-2026"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

for page in pages:
    page.metadata.update({
        "document_id": DOC_ID,
        "title": "Hypertension in Adults: Diagnosis and Management",
        "version": "NG136",
        "publication_date": "2019-08-28",
        "page_number": page.metadata.get("page", 0) + 1,
    })

print(f"Loaded {len(pages)} pages")
print(pages[0].metadata)

C:\Users\ALNOUR\AppData\Local\Temp\ipykernel_11920\660337346.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 52 pages
{'producer': 'Prince 12.5 (www.princexml.com)', 'creator': 'NICE Publications', 'creationdate': '2026-02-26T00:00:00+00:00', 'keywords': 'NG136', 'subject': 'Hypertension in adults: diagnosis and management (NG136)', 'author': 'National Institute for Health and Care Excellence (NICE)', 'title': 'Hypertension in Adults: Diagnosis and Management', 'source': 'assets/hypertension-in-adults-diagnosis-and-management.pdf', 'total_pages': 52, 'page': 0, 'page_label': '1', 'document_id': 'NICE-NG136-2026', 'version': 'NG136', 'publication_date': '2019-08-28', 'page_number': 1}


In [7]:
import os

def validate_pdf(pdf_path: str) -> None:
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"File not found: {pdf_path}")

    if not pdf_path.lower().endswith(".pdf"):
        raise ValueError(f"File is not a PDF: {pdf_path}")

    with open(pdf_path, "rb") as f:
        header = f.read(5)
        if header != b"%PDF-":
            raise ValueError(f"File does not have a valid PDF header: {pdf_path}")

    file_size = os.path.getsize(pdf_path)
    if file_size == 0:
        raise ValueError(f"File is empty: {pdf_path}")

    print(f"PDF validation passed: {pdf_path}")
    print(f"File size: {file_size / 1024:.2f} KB")


def validate_pages_for_chunking(pages) -> None:
    if len(pages) == 0:
        raise ValueError("No pages were loaded from the PDF")

    empty_pages = [
        page.metadata.get("page_number", i)
        for i, page in enumerate(pages, start=1)
        if not page.page_content.strip()
    ]

    total_chars = sum(len(page.page_content) for page in pages)

    if total_chars == 0:
        raise ValueError(
            "No extractable text found in the PDF. "
            "The file may be a scanned/image-based PDF that requires OCR."
        )

    if len(empty_pages) == len(pages):
        raise ValueError("All pages are empty, cannot proceed with chunking")

    print(f"Pages validation passed: {len(pages)} pages")
    print(f"Total extracted characters: {total_chars}")
    if empty_pages:
        print(f"Warning: {len(empty_pages)} empty pages found at: {empty_pages}")


validate_pdf(PDF_PATH)
validate_pages_for_chunking(pages)

PDF validation passed: assets/hypertension-in-adults-diagnosis-and-management.pdf
File size: 287.73 KB
Pages validation passed: 52 pages
Total extracted characters: 98391


In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=850,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(pages)

for i, chunk in enumerate(chunks, start=1):
    chunk.metadata["chunk_id"] = f"{DOC_ID}-CH-{i:03d}"

print(f"Created {len(chunks)} chunks")
print(chunks[0].metadata)

Created 156 chunks
{'producer': 'Prince 12.5 (www.princexml.com)', 'creator': 'NICE Publications', 'creationdate': '2026-02-26T00:00:00+00:00', 'keywords': 'NG136', 'subject': 'Hypertension in adults: diagnosis and management (NG136)', 'author': 'National Institute for Health and Care Excellence (NICE)', 'title': 'Hypertension in Adults: Diagnosis and Management', 'source': 'assets/hypertension-in-adults-diagnosis-and-management.pdf', 'total_pages': 52, 'page': 0, 'page_label': '1', 'document_id': 'NICE-NG136-2026', 'version': 'NG136', 'publication_date': '2019-08-28', 'page_number': 1, 'chunk_id': 'NICE-NG136-2026-CH-001'}


In [9]:
def validate_chunks(chunks) -> None:
    if len(chunks) == 0:
        raise ValueError("No chunks were created from the document")

    chunk_lengths = [len(chunk.page_content) for chunk in chunks]

    empty_chunks = [
        chunk.metadata.get("chunk_id", i)
        for i, chunk in enumerate(chunks, start=1)
        if not chunk.page_content.strip()
    ]

    if empty_chunks:
        raise ValueError(f"Found empty chunks: {empty_chunks}")

    too_large = [
        chunk.metadata.get("chunk_id", i)
        for i, chunk in enumerate(chunks, start=1)
        if len(chunk.page_content) > 2000
    ]

    missing_metadata = [
        chunk.metadata.get("chunk_id", i)
        for i, chunk in enumerate(chunks, start=1)
        if not chunk.metadata.get("document_id") or not chunk.metadata.get("page_number")
    ]

    if missing_metadata:
        raise ValueError(f"Chunks missing required metadata: {missing_metadata}")

    print(f"Chunks validation passed: {len(chunks)} chunks")
    print(f"Average chunk length: {sum(chunk_lengths) / len(chunk_lengths):.0f} characters")
    print(f"Min length: {min(chunk_lengths)}, Max length: {max(chunk_lengths)}")
    if too_large:
        print(f"Warning: {len(too_large)} chunks exceed 2000 characters: {too_large}")


validate_chunks(chunks)

Chunks validation passed: 156 chunks
Average chunk length: 699 characters
Min length: 141, Max length: 849


## Saving embeddings in VectorBD

In [10]:
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings
from langchain_chroma import Chroma

embedding_model = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="hypertension_clinical_kb",
    persist_directory="chroma_db",
    collection_metadata={"hnsw:space": "cosine"}
)

print("Vectorstore created successfully")
print(f"Collection: hypertension_clinical_kb")
print(f"Total chunks stored: {vectorstore._collection.count()}")

c:\Users\ALNOUR\anaconda3\envs\sic\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
c:\Users\ALNOUR\anaconda3\envs\sic\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ALNOUR\AppData\Local\Temp\fastembed_cache\models--qdrant--bge-small-en-v1.5-onnx-q. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details

Vectorstore created successfully
Collection: hypertension_clinical_kb
Total chunks stored: 156


In [11]:
results = vectorstore.similarity_search("What is the target blood pressure?", k=3)
for r in results:
    print(r.metadata["chunk_id"], "-", r.page_content[:100])

NICE-NG136-2026-CH-119 - Blood pressure targets for people with cardiovascular disease 
Recommendation 1.4.23 
Why the commit
NICE-NG136-2026-CH-110 - targets should be used. The committee agreed that in the absence of evidence the focus 
should be on
NICE-NG136-2026-CH-044 - below 140/90 mmHg and ensure that it is maintained below that level. See also 
table 1 for guidance 


## LLM

In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="llama-3.3-70b-versatile",
    base_url="https://api.groq.com/openai/v1",
    api_key=GROQ_API_KEY,
    temperature=0.1,
    max_tokens=700
)

response = llm.invoke("Say 'connection successful' if you can read this.")
print(response.content)

Connection successful.


## RAG Chain

In [14]:
from langchain_core.prompts import ChatPromptTemplate


SYSTEM_PROMPT = """You are a clinical education assistant.
Use ONLY the supplied context. If the context is insufficient, say: "The provided document does not contain enough information to answer that."
Do not diagnose, prescribe, recommend drug doses, or select personalized treatment.
For concerning symptoms, advise assessment by a qualified clinician.
Every factual paragraph must end with one or more citations exactly in this format: [Document ID | p. X | Chunk ID].
Keep the answer clear and concise. End with: "Educational information only; not a diagnosis or medical advice.\""""


prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])

In [15]:
def format_docs(docs):
    blocks = []
    for d in docs:
        m = d.metadata
        citation = f"[{m['document_id']} | p. {m['page_number']} | {m['chunk_id']}]"
        blocks.append(f"SOURCE {citation}\n{d.page_content}")
    return "\n\n".join(blocks)

In [16]:
def ask_clinical_rag(question: str, k: int = 4):
    docs = vectorstore.similarity_search(question, k=k)
    context = format_docs(docs)

    messages = prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)

    return {
        "answer": response.content,
        "retrieved_sources": [
            {
                "document_id": d.metadata["document_id"],
                "page": d.metadata["page_number"],
                "chunk_id": d.metadata["chunk_id"],
                "preview": d.page_content[:180].replace("\n", " ")
            }
            for d in docs
        ]
    }

In [ ]:
result = ask_clinical_rag("What are the target blood pressure readings for people with type 2 diabetes?")
print(result["answer"])
print("\nSources:")
for s in result["retrieved_sources"]:
    print(s)

The target blood pressure reading for people with type 2 diabetes is below 140/90 mmHg. However, previous recommendations suggested a blood pressure target below 130/80 mmHg in the presence of target organ damage such as kidney, cerebrovascular or eye disease [NICE-NG136-2026 | p. 38 | NICE-NG136-2026-CH-113]. 

Educational information only; not a diagnosis or medical advice.

Sources:
{'document_id': 'NICE-NG136-2026', 'page': 14, 'chunk_id': 'NICE-NG136-2026-CH-038', 'preview': 'hypertension in pregnancy.  See also table 1 for clinic blood pressure targets for people aged under 80 and table 2 for  clinic blood pressure targets for people aged 80 and over. '}
{'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-113', 'preview': 'on people already receiving treatment and that it lacked information on adverse events.  The committee agreed that there was no evidence to suggest that blood pressure targets  sho'}
{'document_id': 'NICE-NG136-2026', 'page': 40, 'chu

In [18]:
result = ask_clinical_rag("Is it true that a blood pressure of 200/120 mmHg is considered normal and safe?")
print(result["answer"])
print("\nSources:")
for s in result["retrieved_sources"]:
    print(s)

No, a blood pressure of 200/120 mmHg is not considered normal and safe. According to the provided context, if clinic blood pressure is between 140/90 mmHg and 180/120 mmHg, ambulatory blood pressure monitoring (ABPM) should be offered to confirm the diagnosis of hypertension. A blood pressure of 200/120 mmHg exceeds the upper limit of this range, indicating potential hypertension [NICE-NG136-2026 | p. 7 | NICE-NG136-2026-CH-019]. 

Educational information only; not a diagnosis or medical advice.

Sources:
{'document_id': 'NICE-NG136-2026', 'page': 40, 'chunk_id': 'NICE-NG136-2026-CH-119', 'preview': 'Blood pressure targets for people with cardiovascular disease  Recommendation 1.4.23  Why the committee made the recommendation  The evidence did not show a robust or consistent cl'}
{'document_id': 'NICE-NG136-2026', 'page': 16, 'chunk_id': 'NICE-NG136-2026-CH-044', 'preview': 'below 140/90 mmHg and ensure that it is maintained below that level. See also  table 1 for guidance on clinic bl

In [20]:
result2 = ask_clinical_rag("where is mosalah now?")
print(result2["answer"])
print("\nSources:")
for s in result2["retrieved_sources"]:
    print(s)

The provided document does not contain enough information to answer that. [NICE-NG136-2026 | p. 11 | NICE-NG136-2026-CH-031] 
Educational information only; not a diagnosis or medical advice.

Sources:
{'document_id': 'NICE-NG136-2026', 'page': 52, 'chunk_id': 'NICE-NG136-2026-CH-156', 'preview': '© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-and- conditions#notice-of-rights). Page 52 of 52'}
{'document_id': 'NICE-NG136-2026', 'page': 43, 'chunk_id': 'NICE-NG136-2026-CH-132', 'preview': '© NICE 2026. All rights reserved. Subject to Notice of rights (https://www.nice.org.uk/terms-and- conditions#notice-of-rights). Page 43 of 52'}
{'document_id': 'NICE-NG136-2026', 'page': 11, 'chunk_id': 'NICE-NG136-2026-CH-031', 'preview': 'tobacco. [2004]  1.4.8 Inform people about local initiatives by, for example, healthcare teams or patient  organisations that provide support and promote healthy lifestyle change, '}
{'document_id': 'NICE-NG136-2026', 'p

# ------ 

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()

from controllers.ProcessController import ProcessController

config = {
    "GROQ_API_KEY": os.getenv("GROQ_API_KEY"),
    "GENERATION_MODEL": "openai/gpt-oss-120b",
    "EMBEDDING_MODEL": "BAAI/bge-small-en-v1.5",
    "VECTOR_DB_PATH": "chroma_db",
}

controller = ProcessController(config)


In [12]:
controller.load_pdf(
    pdf_path="assets/hypertension-in-adults-diagnosis-and-management.pdf",
    document_id="NICE-NG136-2026",
    title="Hypertension in Adults: Diagnosis and Management",
    version="NG136",
    publication_date="2019-08-28"
)

controller.chunk_documents(document_id="NICE-NG136-2026")

controller.build_vectorstore(collection_name="hypertension_clinical_kb")

print("Pipeline completed successfully")
print(f"Pages loaded: {len(controller.pages)}")
print(f"Chunks created: {len(controller.chunks)}")

Pipeline completed successfully
Pages loaded: 52
Chunks created: 156


In [13]:
result = controller.ask("What is the target blood pressure for people with type 2 diabetes?")
print(result["answer"])
print("\nSources:")
for s in result["retrieved_sources"]:
    print(s)

The guideline notes that, for people with type 2 diabetes who have target‑organ damage (e.g., kidney, cerebrovascular or eye disease), the recommended blood‑pressure target is **below 130 mm Hg systolic and 80 mm Hg diastolic**【NICE‑NG136‑2026 | p. 38 | NICE‑NG136‑2026‑CH‑113】.  

Educational information only; not a diagnosis or medical advice.

Sources:
{'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-113', 'preview': 'on people already receiving treatment and that it lacked information on adverse events.  The committee agreed that there was no evidence to suggest that blood pressure targets  sho'}
{'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-113', 'preview': 'on people already receiving treatment and that it lacked information on adverse events.  The committee agreed that there was no evidence to suggest that blood pressure targets  sho'}
{'document_id': 'NICE-NG136-2026', 'page': 38, 'chunk_id': 'NICE-NG136-2026-CH-113'